# Analisis Geometris D_min Relasi M3-M2 pada Citra CBCT

Notebook ini memproses segmentasi volumetrik 3D (.mha) untuk menghitung jarak minimum fisik ($D_{\min}$) antarpermukaan gigi molar kedua (M2) dan molar ketiga (M3).

### Alur Kerja:
1. Konfigurasi path folder dataset dan parameter analisis.
2. Parsing metadata volume dan voxel spacing fisik (mm).
3. Ekstraksi voxel permukaan anatomis via erosi biner 3D (26-connectivity).
4. Penentuan jarak minimum fisik ($D_{\min}$) menggunakan 3D KD-Tree Euclidean query.
5. Klasifikasi rule-based awal (Independent vs Contact/Resorption Zone).
6. Visualisasi irisan sagital beranotasi dan ekspor ringkasan hasil ke CSV.

## 1. Konfigurasi Input dan Parameter
Tentukan path folder yang memuat file mask segmentasi (.mha).

In [ ]:
from pathlib import Path

# Path folder mask segmentasi (.mha)
DATASET_DIR = Path(r"d:/Workspace/TA3/data/toothfairy2/Dataset112_ToothFairy2/labelsTr")

# Folder penyimpanan keluaran CSV dan visualisasi
OUTPUT_DIR = Path(r"d:/Workspace/TA3/data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Pasangan FDI gigi yang dianalisis
# Rahang atas (Maxillary): (17, 18) dan (27, 28)
# Rahang bawah (Mandibular): (37, 38) dan (47, 48)
TARGET_PAIRS = [(17, 18), (27, 28)]

# Ambang batas pemisahan fisik (mm)
DMIN_THRESHOLD_MM = 0.50

# Batas jumlah berkas yang diproses (atur None untuk memproses seluruh berkas)
MAX_FILES = 20

## 2. Definisi Modul Geometri dan I/O

In [ ]:
import time
import csv
import numpy as np
from scipy import ndimage
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def read_mha(filepath: Path) -> tuple[dict, np.ndarray]:
    """
    Membaca berkas MetaImage (.mha) tanpa dependensi eksternal.
    Mengembalikan metadata header dan array numpy 3D (Z, Y, X).
    """
    meta = {}
    with open(filepath, "rb") as f:
        while True:
            line = f.readline()
            if not line:
                break
            line_str = line.decode("ascii", errors="ignore").strip()
            if "=" in line_str:
                k, v = line_str.split("=", 1)
                meta[k.strip()] = v.strip()
            if b"ElementDataFile" in line:
                break
        dim = [int(x) for x in meta["DimSize"].split()]
        data = np.fromfile(f, dtype=np.uint8)
        arr = data.reshape((dim[2], dim[1], dim[0]))
    return meta, arr

def get_spacing_zyx(meta: dict) -> np.ndarray:
    """
    Mengekstrak voxel spacing fisik dalam urutan sumbu (Z, Y, X) satuan mm/voxel.
    """
    sp = [float(x) for x in meta["ElementSpacing"].split()]
    return np.array([sp[2], sp[1], sp[0]])

def get_surface_voxels(mask_3d: np.ndarray) -> np.ndarray:
    """
    Mengekstrak voxel permukaan luar mask biner via erosi 3D dengan 26-connectivity.
    """
    struct = ndimage.generate_binary_structure(3, 3)
    eroded = ndimage.binary_erosion(mask_3d, structure=struct)
    surface = mask_3d ^ eroded
    return np.argwhere(surface)

def compute_dmin(mask_m2: np.ndarray, mask_m3: np.ndarray, spacing_zyx: np.ndarray):
    """
    Menghitung jarak minimum Euclidean fisik (mm) antarpermukaan mask M2 dan M3.
    """
    sv_m2 = get_surface_voxels(mask_m2)
    sv_m3 = get_surface_voxels(mask_m3)

    phys_m2 = sv_m2 * spacing_zyx
    phys_m3 = sv_m3 * spacing_zyx

    tree = cKDTree(phys_m2)
    dists, idx_m2 = tree.query(phys_m3)

    best = int(np.argmin(dists))
    dist_mm = float(dists[best])
    coord_m3 = sv_m3[best]
    coord_m2 = sv_m2[idx_m2[best]]

    return dist_mm, coord_m2, coord_m3

## 3. Eksekusi Batch Scanner
Menjalankan pemindaian pada berkas segmentasi di `DATASET_DIR`.

In [ ]:
files = sorted(list(DATASET_DIR.glob("*.mha")))
if MAX_FILES is not None:
    files = files[:MAX_FILES]

results = []
total_files = len(files)

print(f"Memproses {total_files} berkas dari {DATASET_DIR}...")

for idx, fp in enumerate(files, 1):
    try:
        meta, arr = read_mha(fp)
        spacing_zyx = get_spacing_zyx(meta)
    except Exception as e:
        print(f"[{idx}/{total_files}] Gagal membaca {fp.name}: {e}")
        continue

    for m2_id, m3_id in TARGET_PAIRS:
        mask_m2 = (arr == m2_id)
        mask_m3 = (arr == m3_id)

        m2_count = int(np.sum(mask_m2))
        m3_count = int(np.sum(mask_m3))

        if m2_count == 0 or m3_count == 0:
            status = "ABSENT_BOTH" if (m2_count == 0 and m3_count == 0) else ("ABSENT_M2" if m2_count == 0 else "ABSENT_M3")
            results.append({
                "file": fp.name,
                "pair": f"{m2_id}-{m3_id}",
                "m2_voxels": m2_count,
                "m3_voxels": m3_count,
                "dmin_mm": None,
                "class": status,
                "coord_m2": None,
                "coord_m3": None,
            })
            continue

        dist_mm, c2, c3 = compute_dmin(mask_m2, mask_m3, spacing_zyx)
        category = "INDEPENDENT" if dist_mm > DMIN_THRESHOLD_MM else "CONTACT_OR_RESORPTION"

        results.append({
            "file": fp.name,
            "pair": f"{m2_id}-{m3_id}",
            "m2_voxels": m2_count,
            "m3_voxels": m3_count,
            "dmin_mm": round(dist_mm, 4),
            "class": category,
            "coord_m2": c2.tolist(),
            "coord_m3": c3.tolist(),
        })

print("Pemrosesan selesai.")

## 4. Ringkasan Statistik dan Ekspor CSV

In [ ]:
valid = [r for r in results if r["dmin_mm"] is not None]
absent = [r for r in results if r["dmin_mm"] is None]
indep = [r for r in valid if r["class"] == "INDEPENDENT"]
contact = [r for r in valid if r["class"] == "CONTACT_OR_RESORPTION"]
dmins = [r["dmin_mm"] for r in valid]

print("=" * 60)
print("RINGKASAN HASIL PERHITUNGAN D_MIN")
print("=" * 60)
print(f"Total pasangan dievaluasi   : {len(results)}")
print(f"Gigi tidak lengkap / absen : {len(absent)}")
print(f"Pasangan terukur valid      : {len(valid)}")
print("")
if len(valid) > 0:
    print(f"INDEPENDENT (> {DMIN_THRESHOLD_MM:.2f} mm)        : {len(indep)} ({len(indep)/len(valid)*100:.1f}%)")
    print(f"CONTACT/RESORPTION (<= {DMIN_THRESHOLD_MM:.2f} mm) : {len(contact)} ({len(contact)/len(valid)*100:.1f}%)")
    print("")
    print(f"D_min minimum : {min(dmins):.4f} mm")
    print(f"D_min maximum : {max(dmins):.4f} mm")
    print(f"D_min mean    : {np.mean(dmins):.4f} mm")
    print(f"D_min median  : {np.median(dmins):.4f} mm")
    print(f"D_min std     : {np.std(dmins):.4f} mm")
print("=" * 60)

# Ekspor CSV
csv_out = OUTPUT_DIR / "dmin_summary.csv"
with open(csv_out, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["file", "pair", "m2_voxels", "m3_voxels", "dmin_mm", "class", "coord_m2", "coord_m3"])
    writer.writeheader()
    writer.writerows(results)

print(f"Hasil disimpan ke: {csv_out}")

## 5. Visualisasi Irisan Sagital pada Kasus Tertentu
Tampilkan irisan sagital yang melalui titik kontak terdekat pada pasangan gigi yang terdeteksi.

In [ ]:
if len(valid) > 0:
    sample = valid[0]
    sample_file = DATASET_DIR / sample["file"]
    meta, arr = read_mha(sample_file)

    c2 = sample["coord_m2"]
    c3 = sample["coord_m3"]
    x_slice = int((c2[2] + c3[2]) // 2)

    slice_2d = arr[:, :, x_slice]
    m2_id, m3_id = [int(x) for x in sample["pair"].split("-")]

    # Visualisasi RGB mask
    h, w = slice_2d.shape
    canvas = np.zeros((h, w, 3), dtype=np.float32)

    canvas[slice_2d == m2_id] = [0.2, 0.6, 1.0]  # M2 (Biru)
    canvas[slice_2d == m3_id] = [1.0, 0.45, 0.1] # M3 (Oranye)
    other = (slice_2d > 0) & (slice_2d != m2_id) & (slice_2d != m3_id)
    canvas[other] = [0.6, 0.6, 0.6]

    plt.figure(figsize=(8, 6))
    plt.imshow(canvas, origin="upper")
    plt.plot([c2[1], c3[1]], [c2[0], c3[0]], "r--", linewidth=1.5)
    plt.plot(c2[1], c2[0], "o", color="#38bdf8", markersize=6, label=f"M2 (#{m2_id})")
    plt.plot(c3[1], c3[0], "o", color="#fb923c", markersize=6, label=f"M3 (#{m3_id})")

    plt.title(f"{sample['file']} [{sample['pair']}] | D_min: {sample['dmin_mm']:.4f} mm | Status: {sample['class']}\nSagittal Slice X={x_slice}", fontsize=10)
    plt.axis("off")
    plt.legend(loc="lower left")
    plt.tight_layout()
    plt.show()
else:
    print("Tidak ada pasangan valid yang dapat divisualisasikan.")